# Tüm sonuçlar — tablolar ve grafikleri dışa aktar

Bu notebook, `1_baseline_cnn`, `2_baseline_vit`, `3_optimized_cnn` ve `4_optimized_vit` notebook'larının Drive'a kaydettiği sonuçları okur. Bütün tabloları ve grafikleri tek bir klasöre ve zip'e aktarır. GPU gerekmez; bazı gruplar henüz bitmemişse de çalışır, o ana kadar tamamlanan fold'ları kullanır.

**Çıktılar** (Drive: `breast_mri_standard_results/holdout5/EXPORT/` ve `EXPORT_holdout5.zip`):

| Klasör | İçerik |
|---|---|
| `tablolar/ALL_RESULTS.xlsx` | Test / Validation / Train ortalama ± std tabloları (makale tabloları), Baseline→Optimize kazancı, 5 fold ortalama confusion matrix, eğitim süreleri, veri bölünmesi, sızıntı kontrolü, tamamlanma durumu ve ham veriler. Aynı tablolar CSV olarak da kaydedilir. |
| `grafikler/1_karsilastirma` | Her metrik için Baseline ve Optimize çubuk grafiği (CNN ve ViT yan yana); her grup için Train / Validation / Test doğruluğu |
| `grafikler/2_ortalama_confusion` | Her grup ve split için 5 fold ortalama ± std confusion matrix |
| `grafikler/3_ortalama_roc` | Her grup için 5 fold ortalama test ROC eğrileri |
| `grafikler/4_ortalama_ogrenme_egrileri` | Accuracy ve loss eğrileri (5 fold ort. ± std) |
| `grafikler/5_fold_bazli` | Her model ve fold için 11 grafik ve CSV dosyaları |

Bütün grafikler 300 dpi PNG olarak kaydedilir. Zip 500 MB'tan küçükse otomatik olarak indirilir.


In [ ]:
# ======================= 1. AYARLAR + SONUÇLARI OKU =======================
# Dört grup notebook'u (1–4) tamamlandıktan sonra çalıştır. GPU gerekmez.
# Bazı gruplar bitmemişse de çalışır; o ana kadar tamamlanan fold'ları kullanır.
from pathlib import Path
import json, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc
from IPython.display import display
from google.colab import drive
drive.mount("/content/drive")

OUTPUT_ROOT = Path("/content/drive/MyDrive/breast_mri_standard_results")   # grup notebook'larıyla aynı olmalı
CV_MODE = "holdout5"                                                       # grup notebook'larıyla aynı olmalı
COPY_FOLD_FIGURES = True     # her fold'un 11 grafiğini de dışa aktarım klasörüne kopyala
SHOW = True                  # karşılaştırma grafiklerini hücre altında da göster

GROUPS = ["baseline_cnn", "baseline_vit", "optimized_cnn", "optimized_vit"]
GROUP_TR = {"baseline_cnn": "Baseline CNN", "baseline_vit": "Baseline ViT",
            "optimized_cnn": "Optimize CNN", "optimized_vit": "Optimize ViT"}
CNN_ORDER = ["ResNet34", "DenseNet121", "EfficientNet-B0", "MobileNetV2", "InceptionV3"]
VIT_ORDER = ["ViT-B-16", "ViT-B-32", "ViT-L-16", "ViT-L-32", "Swin-T"]
METRIC_COLS = ["Accuracy", "Precision", "Recall (Sensitivity)", "Specificity", "F1-Score", "ROC-AUC", "Loss"]
# Kategorik renkler sabit sırayla (renk körlüğü için doğrulanmış palet); çizgilerde ayrıca çizgi stili de farklı
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
LINESTYLES = ["-", "--", "-.", ":", (0, (5, 1, 1, 1))]
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 10})

ROOT = OUTPUT_ROOT / CV_MODE
EXPORT = ROOT / "EXPORT"
if EXPORT.exists(): shutil.rmtree(EXPORT)
for sub in ["tablolar", "grafikler/1_karsilastirma", "grafikler/2_ortalama_confusion",
            "grafikler/3_ortalama_roc", "grafikler/4_ortalama_ogrenme_egrileri", "grafikler/5_fold_bazli"]:
    (EXPORT / sub).mkdir(parents=True, exist_ok=True)

def fold_dirs():
    for p in sorted(ROOT.glob("*/*/Fold_*/result.json")):
        if "EXPORT" not in p.parts: yield p.parent

results = [json.loads((d / "result.json").read_text()) for d in fold_dirs()]
assert results, f"{ROOT} altında hiç sonuç yok."
status = pd.DataFrame([{"Group": r["Group"], "Model": r["Model"], "Fold": r["Fold"]} for r in results])
progress = status.groupby(["Group", "Model"]).Fold.count().rename("Tamamlanan fold").reset_index()
display(progress)
if (progress["Tamamlanan fold"] < 5).any(): print("UYARI: 5 fold'u tamamlanmamış modeller var (yukarıda < 5 olanlar).")
for g in GROUPS:
    if g not in set(status.Group): print(f"UYARI: {GROUP_TR[g]} grubunun henüz hiç sonucu yok.")

def model_order(models):
    return [m for m in CNN_ORDER + VIT_ORDER if m in models]

def save(fig, path):
    fig.tight_layout(); fig.savefig(path, bbox_inches="tight")
    plt.show() if SHOW else None
    plt.close(fig)


In [ ]:
# ======================= 2. TABLOLAR (Excel + CSV) =======================
rows = [{"Group": r["Group"], "Grup": GROUP_TR[r["Group"]], "Model": r["Model"], "Fold": r["Fold"],
         "Best Epoch": r["Best Epoch"], "Train Time (s)": r.get("Train Time (s)"), "Split": s, **m}
        for r in results for s, m in r["Splits"].items()]
per_fold = pd.DataFrame(rows)
keys = ["Group", "Grup", "Model", "Split"]
g = per_fold.groupby(keys, sort=False)
summary = g[METRIC_COLS].mean().add_suffix(" mean").join(g[METRIC_COLS].std(ddof=1).add_suffix(" std")).reset_index()
article = g[METRIC_COLS[:-1]].agg(lambda s: f"{100*s.mean():.2f} ± {100*s.std(ddof=1):.2f}").reset_index()
cm_mean = g[["TN", "FP", "FN", "TP"]].mean().round(1).reset_index()          # 5 fold ortalama confusion matrix
time_tbl = per_fold[per_fold.Split == "Test"].groupby(["Grup", "Model"], sort=False)["Train Time (s)"].agg(
    lambda s: f"{s.mean()/60:.1f} ± {s.std(ddof=1)/60:.1f} dk").reset_index(name="Eğitim süresi (fold başı)")

order = {g: i for i, g in enumerate(GROUPS)}; morder = {m: i for i, m in enumerate(CNN_ORDER + VIT_ORDER)}
def sort(df):
    return df.sort_values(["Group", "Model"], key=lambda c: c.map(order) if c.name == "Group" else c.map(morder))
article, summary, cm_mean = sort(article), sort(summary), sort(cm_mean)

# Baseline → Optimize iyileşmesi (test, ortalama; yüzde puan)
test_mean = summary[summary.Split == "Test"].set_index(["Group", "Model"])
gain_rows = []
for fam in ["cnn", "vit"]:
    for m in model_order(set(test_mean.index.get_level_values(1))):
        b, o = (f"baseline_{fam}", m), (f"optimized_{fam}", m)
        if b in test_mean.index and o in test_mean.index:
            gain_rows.append({"Aile": fam.upper(), "Model": m, **{
                f"{k} (Δ puan)": round(100 * (test_mean.loc[o, f"{k} mean"] - test_mean.loc[b, f"{k} mean"]), 2)
                for k in ["Accuracy", "F1-Score", "ROC-AUC", "Recall (Sensitivity)", "Specificity"]}})
gain = pd.DataFrame(gain_rows)

fold_counts_file = OUTPUT_ROOT / f"fold_counts_{CV_MODE}.csv"
leak_file = OUTPUT_ROOT / "leakage_check.csv"
xlsx = EXPORT / "tablolar" / "ALL_RESULTS.xlsx"
with pd.ExcelWriter(xlsx, engine="openpyxl") as xw:
    for split in ["Test", "Validation", "Train"]:
        t = article[article.Split == split].drop(columns=["Group", "Split"])
        t.to_excel(xw, sheet_name=f"{split} (ort±std)", index=False)
        t.to_csv(EXPORT / "tablolar" / f"{split.lower()}_ort_std.csv", index=False)
    if len(gain): gain.to_excel(xw, sheet_name="Baseline→Optimize kazanç", index=False)
    cm_mean.drop(columns=["Group"]).to_excel(xw, sheet_name="Ortalama CM", index=False)
    time_tbl.to_excel(xw, sheet_name="Eğitim süresi", index=False)
    if fold_counts_file.exists(): pd.read_csv(fold_counts_file).to_excel(xw, sheet_name="Veri bölünmesi", index=False)
    if leak_file.exists(): pd.read_csv(leak_file).to_excel(xw, sheet_name="Sızıntı kontrolü", index=False)
    progress.to_excel(xw, sheet_name="Tamamlanma durumu", index=False)
    summary.drop(columns=["Group"]).to_excel(xw, sheet_name="Ortalama-Std (ham)", index=False)
    per_fold.drop(columns=["Group"]).to_excel(xw, sheet_name="Fold bazlı (ham)", index=False)
    for ws in xw.sheets.values():          # sütun genişlikleri
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = min(40, max(10, max(len(str(c.value or "")) for c in col) + 2))
per_fold.to_csv(EXPORT / "tablolar" / "fold_bazli_ham.csv", index=False)
print("Kaydedildi:", xlsx)
display(article[article.Split == "Test"].drop(columns=["Group", "Split"]))
if len(gain): display(gain)


In [ ]:
# ======================= 3. KARŞILAŞTIRMA GRAFİKLERİ =======================
OUTC = EXPORT / "grafikler" / "1_karsilastirma"
COLOR_B, COLOR_O = PALETTE[0], PALETTE[1]     # Baseline = mavi, Optimize = turuncu (her grafikte aynı)

def bar_compare(metric, split="Test"):
    """Her aile (CNN, ViT) için model başına Baseline ve Optimize çubukları, hata çubuğu = std."""
    s = summary[summary.Split == split].set_index(["Group", "Model"])
    fams = [f for f in ["cnn", "vit"] if any(g.endswith(f) for g in s.index.get_level_values(0))]
    fig, axes = plt.subplots(1, len(fams), figsize=(6.2 * len(fams), 4.2), squeeze=False, sharey=True)
    for ax, fam in zip(axes[0], fams):
        models = model_order({m for g, m in s.index if g.endswith(fam)}); x = np.arange(len(models)); w = 0.38
        for k, (grp, col, lab) in enumerate([(f"baseline_{fam}", COLOR_B, "Baseline"), (f"optimized_{fam}", COLOR_O, "Optimize")]):
            mean = [100 * s.loc[(grp, m), f"{metric} mean"] if (grp, m) in s.index else np.nan for m in models]
            std = [100 * s.loc[(grp, m), f"{metric} std"] if (grp, m) in s.index else 0 for m in models]
            bars = ax.bar(x + (k - .5) * w, mean, w - 0.03, yerr=std, capsize=3, color=col, label=lab,
                          error_kw={"elinewidth": 1, "ecolor": "#555"})
            for b, v, e in zip(bars, mean, std):
                if not np.isnan(v):   # değer etiketi hata çubuğunun üstünde
                    ax.text(b.get_x() + b.get_width() / 2, v + e + 0.5, f"{v:.1f}", ha="center", va="bottom", fontsize=7.5, color="#333")
        ax.set_xticks(x); ax.set_xticklabels(models, rotation=20, ha="right")
        ax.set_title(f"{fam.upper()} modelleri"); ax.grid(axis="x", visible=False)
        lo = np.nanmin([100 * s[f"{metric} mean"].min()]) - 8
        ax.set_ylim(max(0, lo), 104)
    axes[0][0].set_ylabel(f"{metric} (%) — {split}, 5 fold ort. ± std")
    h, l = axes[0][0].get_legend_handles_labels()
    fig.suptitle(f"Baseline ve Optimize — {split} {metric}", fontsize=12, y=1.04)
    fig.legend(h, l, loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 1.0))
    save(fig, OUTC / f"{split.lower()}_{metric.split()[0].lower().replace('-', '')}_baseline_vs_optimize.png")

for metric in ["Accuracy", "F1-Score", "ROC-AUC", "Recall (Sensitivity)", "Specificity", "Precision"]:
    bar_compare(metric, "Test")
bar_compare("Accuracy", "Validation")

# Train / Validation / Test doğruluğu yan yana (aşırı öğrenme kontrolü) — her grup için
for grp in [g for g in GROUPS if g in set(summary.Group)]:
    s = summary[summary.Group == grp]; models = model_order(set(s.Model)); x = np.arange(len(models)); w = 0.27
    fig, ax = plt.subplots(figsize=(7.5, 4))
    for k, (split, col) in enumerate(zip(["Train", "Validation", "Test"], PALETTE[:3])):
        r = s[s.Split == split].set_index("Model")
        ax.bar(x + (k - 1) * w, [100 * r.loc[m, "Accuracy mean"] for m in models], w - 0.03,
               yerr=[100 * r.loc[m, "Accuracy std"] for m in models], capsize=2, color=col, label=split,
               error_kw={"elinewidth": 1, "ecolor": "#555"})
    ax.set_xticks(x); ax.set_xticklabels(models, rotation=20, ha="right"); ax.grid(axis="x", visible=False)
    ax.set_ylim(max(0, 100 * s["Accuracy mean"].min() - 8), 101.5); ax.set_ylabel("Accuracy (%) — 5 fold ort. ± std")
    ax.set_title(f"{GROUP_TR[grp]} — Train / Validation / Test doğruluğu", pad=26)
    ax.legend(frameon=False, ncol=3, loc="lower center", bbox_to_anchor=(0.5, 1.0))
    save(fig, OUTC / f"{grp}_train_val_test_accuracy.png")


In [ ]:
# ======================= 4. ORTALAMA CONFUSION MATRIX, ROC VE ÖĞRENME EĞRİLERİ =======================
by_key = {(r["Group"], r["Model"]): [] for r in results}
for d in fold_dirs():
    r = json.loads((d / "result.json").read_text()); by_key[(r["Group"], r["Model"])].append(d)

# 4a) Ortalama confusion matrix (5 fold ortalaması ± std), her grup ve split için tek figür
for grp in [g for g in GROUPS if g in set(per_fold.Group)]:
    models = model_order(set(per_fold[per_fold.Group == grp].Model))
    for split in ["Train", "Validation", "Test"]:
        fig, axes = plt.subplots(1, len(models), figsize=(3.3 * len(models), 3.2), squeeze=False)
        for ax, m in zip(axes[0], models):
            d = per_fold[(per_fold.Group == grp) & (per_fold.Model == m) & (per_fold.Split == split)]
            mean = np.array([[d.TN.mean(), d.FP.mean()], [d.FN.mean(), d.TP.mean()]])
            std = np.array([[d.TN.std(ddof=1), d.FP.std(ddof=1)], [d.FN.std(ddof=1), d.TP.std(ddof=1)]])
            annot = np.array([[f"{mean[i, j]:.0f}\n±{std[i, j]:.0f}" for j in range(2)] for i in range(2)])
            sns.heatmap(mean, annot=annot, fmt="", cmap="Blues", cbar=False, ax=ax, square=True,
                        xticklabels=["Benign", "Malignant"], yticklabels=["Benign", "Malignant"], annot_kws={"fontsize": 9})
            ax.grid(False)
            ax.set_title(f"{m}\nN={int(d.N.mean())}", fontsize=9); ax.set_xlabel("Tahmin"); ax.set_ylabel("Gerçek" if ax is axes[0][0] else "")
        fig.suptitle(f"{GROUP_TR[grp]} — {split} confusion matrix (5 fold ort. ± std)", fontsize=11)
        save(fig, EXPORT / "grafikler" / "2_ortalama_confusion" / f"{grp}_{split.lower()}_cm.png")

# 4b) Ortalama test ROC eğrisi (fold'lar arası interpolasyonla), her grup için tüm modeller tek grafikte
grid = np.linspace(0, 1, 201)
for grp in [g for g in GROUPS if g in set(per_fold.Group)]:
    models = model_order({m for g, m in by_key if g == grp})
    fig, ax = plt.subplots(figsize=(5.6, 5))
    for i, m in enumerate(models):
        tprs, aucs = [], []
        for d in by_key[(grp, m)]:
            p = d / "test_predictions.csv"
            if not p.exists(): continue
            t = pd.read_csv(p); fpr, tpr, _ = roc_curve(t.y, t.prob_malignant)
            tp = np.interp(grid, fpr, tpr); tp[0] = 0; tprs.append(tp); aucs.append(auc(fpr, tpr))
        if not tprs: continue
        mt = np.mean(tprs, 0); mt[-1] = 1
        ax.plot(grid, mt, color=PALETTE[i % 5], ls=LINESTYLES[i % 5], lw=2,
                label=f"{m} (AUC {np.mean(aucs):.3f} ± {np.std(aucs, ddof=1) if len(aucs) > 1 else 0:.3f})")
    ax.plot([0, 1], [0, 1], color="#999", lw=1, ls="--")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate"); ax.set_aspect("equal")
    ax.set_title(f"{GROUP_TR[grp]} — Test ROC (5 fold ort.)"); ax.legend(loc="lower right", fontsize=8, frameon=False)
    save(fig, EXPORT / "grafikler" / "3_ortalama_roc" / f"{grp}_test_roc.png")

# 4c) Ortalama öğrenme eğrileri (accuracy ve loss, ± std gölge), her grup için model başına bir panel
for grp in [g for g in GROUPS if g in set(per_fold.Group)]:
    models = model_order({m for g, m in by_key if g == grp})
    for key in ["Accuracy", "Loss"]:
        fig, axes = plt.subplots(1, len(models), figsize=(3.4 * len(models), 3), squeeze=False, sharey=True)
        for ax, m in zip(axes[0], models):
            hs = [pd.read_csv(d / "history.csv") for d in by_key[(grp, m)] if (d / "history.csv").exists()]
            if not hs: continue
            ep = hs[0].Epoch
            for split, col, ls in [("Train", PALETTE[0], "-"), ("Validation", PALETTE[1], "--")]:
                v = np.array([h[f"{split} {key}"].values for h in hs]); mu, sd = v.mean(0), v.std(0)
                ax.plot(ep, mu, color=col, ls=ls, lw=2, marker="o", ms=3, label=split)
                ax.fill_between(ep, mu - sd, mu + sd, color=col, alpha=.15, lw=0)
            ax.set_title(m, fontsize=9); ax.set_xlabel("Epoch")
        axes[0][0].set_ylabel(key); axes[0][-1].legend(frameon=False, fontsize=8)
        fig.suptitle(f"{GROUP_TR[grp]} — {key} eğrileri (5 fold ort. ± std)", fontsize=11)
        save(fig, EXPORT / "grafikler" / "4_ortalama_ogrenme_egrileri" / f"{grp}_{key.lower()}.png")


In [ ]:
# ======================= 5. FOLD GRAFİKLERİNİ KOPYALA + ZIP + İNDİR =======================
if COPY_FOLD_FIGURES:
    n = 0
    for d in fold_dirs():
        dst = EXPORT / "grafikler" / "5_fold_bazli" / d.relative_to(ROOT); dst.mkdir(parents=True, exist_ok=True)
        for f in list(d.glob("*.png")) + list(d.glob("*.csv")):
            shutil.copy2(f, dst / f.name); n += 1
    print(f"{n} fold dosyası kopyalandı (her fold: 11 grafik + metrics/history/test_predictions CSV).")

(EXPORT / "OKU_BENI.txt").write_text(
    "tablolar/ALL_RESULTS.xlsx         : Makale tabloları (Test/Validation/Train ort±std), kazanç, ortalama CM, veri bölünmesi, sızıntı kontrolü, ham veriler\n"
    "grafikler/1_karsilastirma         : Baseline ve Optimize çubuk grafikleri (metrik başına) ve grup başına train/val/test doğruluğu\n"
    "grafikler/2_ortalama_confusion    : Her grup ve split için 5 fold ortalama ± std confusion matrix\n"
    "grafikler/3_ortalama_roc          : Her grup için 5 fold ortalama test ROC eğrileri\n"
    "grafikler/4_ortalama_ogrenme_egrileri : Accuracy ve loss eğrileri (5 fold ort. ± std)\n"
    "grafikler/5_fold_bazli            : Her model ve fold için 11 grafik + CSV dosyaları\n", encoding="utf-8")

zip_path = shutil.make_archive(str(ROOT / f"EXPORT_{CV_MODE}"), "zip", EXPORT)
size_mb = Path(zip_path).stat().st_size / 1e6
print(f"Zip: {zip_path} ({size_mb:.0f} MB) — Drive'da bu klasörde duruyor.")
try:
    from google.colab import files
    if size_mb < 500: files.download(zip_path)
    else: print("Zip büyük; Drive'dan indir.")
except Exception as e:
    print("Otomatik indirme başlatılamadı, Drive'dan indir:", e)
